### فاز اول 

In [ ]:
import os, json, time, pickle, glob
import numpy as np
import torch
from torch.utils.data import DataLoader

from load_data import get_rat_hippo
from model    import RatModel, SharedRNNModel
from utils     import (collate_fn,
                       train_one_epoch, eval_one_epoch,
                       train_one_epoch_shared, eval_one_epoch_shared)

import matplotlib
matplotlib.use('Agg')          
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [4]:
RATS = {
    'achilles': {'file': 'data\\hippo_nn_achilles.npz', 'M': 120},
    'buddy':    {'file': 'data\\hippo_nn_buddy.npz',    'M': 48},
    'cicero':   {'file': 'data\\hippo_nn_cicero.npz',   'M': 55},
    'gatsby':   {'file': 'data\\hippo_nn_gatsby.npz',   'M': 66},
}
SOURCE_RATS = ['achilles', 'buddy', 'cicero']
TARGET_RAT  = 'gatsby'

In [5]:
CFG = dict(
    latent_dim   = 64,
    hidden_dim   = 128,
    num_layers   = 1,
    batch_size   = 8,
    lr           = 1e-3,
    weight_decay = 1e-4,
    epochs_s1    = 60,
    epochs_s2    = 60,
    epochs_pre   = 60,
    epochs_ft    = 60,
    seed         = 42,
    max_trial_len= 500,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs('results/series1', exist_ok=True)
os.makedirs('results/series2', exist_ok=True)
os.makedirs('results/series3', exist_ok=True)

print(f'  Device: {DEVICE}   PyTorch: {torch.__version__}')


  Device: cpu   PyTorch: 2.12.0+cpu


In [6]:
def make_loaders(rat_names, batch_size=8):
    train_l, val_l = {}, {}
    for rat in rat_names:
        info = RATS[rat]
        tr_ds, vl_ds = get_rat_hippo(info['file'], CFG['max_trial_len'])
        train_l[rat] = DataLoader(tr_ds, batch_size=batch_size,
                                  shuffle=True,  collate_fn=collate_fn)
        val_l[rat]   = DataLoader(vl_ds, batch_size=batch_size,
                                  shuffle=False, collate_fn=collate_fn)
    return train_l, val_l

def make_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=8)

def save_json(obj, path):
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)

def log_row(epoch, total, tr_loss, tr_r2, vl_loss, vl_r2, best, t0):
    print(f'  Ep {epoch:3d}/{total} | '
          f'Tr loss={tr_loss:.4f} R²={tr_r2:.4f} | '
          f'Vl loss={vl_loss:.4f} R²={vl_r2:.4f} | '
          f'Best={best:.4f} | {time.time()-t0:.0f}s')



In [7]:
def run_series1():
    
    print('  SERIES 1 — Independent model per rat')
    
    torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])
    summary = {}

    for rnn_type in ['RNN', 'GRU']:
        for rat, info in RATS.items():
            tag = f'{rat}_{rnn_type}'
            print(f'\n── {tag} ──────────────────────────────')
            t0 = time.time()

            tr_l, vl_l = make_loaders([rat], CFG['batch_size'])
            model = RatModel(
                input_dim=info['M'],
                latent_dim=CFG['latent_dim'],
                hidden_dim=CFG['hidden_dim'],
                rnn_type=rnn_type,
                num_layers=CFG['num_layers'],
            ).to(DEVICE)

            opt  = torch.optim.Adam(model.parameters(),
                                    lr=CFG['lr'], weight_decay=CFG['weight_decay'])
            sch  = make_scheduler(opt)
            hist = {'train_loss':[], 'train_r2':[], 'val_loss':[], 'val_r2':[]}
            best_r2   = -np.inf
            best_path = f'results/series1/best_{tag}.pt'

            for ep in range(1, CFG['epochs_s1'] + 1):
                tr_loss, tr_r2 = train_one_epoch(model, tr_l[rat], opt, DEVICE)
                vl_loss, vl_r2 = eval_one_epoch(model,  vl_l[rat], DEVICE)
                sch.step(vl_r2)

                hist['train_loss'].append(tr_loss)
                hist['train_r2'].append(tr_r2)
                hist['val_loss'].append(vl_loss)
                hist['val_r2'].append(vl_r2)

                if vl_r2 > best_r2:
                    best_r2 = vl_r2
                    torch.save(model.state_dict(), best_path)

                if ep % 10 == 0 or ep == 1:
                    log_row(ep, CFG['epochs_s1'], tr_loss, tr_r2, vl_loss, vl_r2, best_r2, t0)

            save_json(hist, f'results/series1/history_{tag}.json')
            summary[tag] = {'best_val_r2': best_r2,
                            'final_train_r2': hist['train_r2'][-1]}
            print(f'  ✓ Best Val R² = {best_r2:.4f}')

    save_json(summary, 'results/series1/summary.json')
    print('\n  ── Series 1 Summary ──')
    for k, v in summary.items():
        print(f'  {k:20s}  Val R²={v["best_val_r2"]:.4f}')
    return summary


In [8]:
run_series1()

  SERIES 1 — Independent model per rat

── achilles_RNN ──────────────────────────────
  [data\hippo_nn_achilles.npz] trials=85 neurons=120 train=68 val=17
  Ep   1/60 | Tr loss=0.5180 R²=-1.0260 | Vl loss=0.3237 R²=-0.4312 | Best=-0.4312 | 5s
  Ep  10/60 | Tr loss=0.0318 R²=0.8742 | Vl loss=0.0329 R²=0.8566 | Best=0.8596 | 9s
  Ep  20/60 | Tr loss=0.0176 R²=0.9301 | Vl loss=0.0254 R²=0.8881 | Best=0.8881 | 12s
  Ep  30/60 | Tr loss=0.0122 R²=0.9514 | Vl loss=0.0241 R²=0.8919 | Best=0.8919 | 16s
  Ep  40/60 | Tr loss=0.0076 R²=0.9699 | Vl loss=0.0235 R²=0.8942 | Best=0.9009 | 19s
  Ep  50/60 | Tr loss=0.0056 R²=0.9773 | Vl loss=0.0218 R²=0.9016 | Best=0.9064 | 23s
  Ep  60/60 | Tr loss=0.0043 R²=0.9828 | Vl loss=0.0232 R²=0.8946 | Best=0.9064 | 27s
  ✓ Best Val R² = 0.9064

── buddy_RNN ──────────────────────────────
  [data\hippo_nn_buddy.npz] trials=58 neurons=48 train=46 val=12
  Ep   1/60 | Tr loss=0.5802 R²=-1.4248 | Vl loss=0.2793 R²=-0.1371 | Best=-0.1371 | 0s
  Ep  10/60 | Tr l

{'achilles_RNN': {'best_val_r2': 0.9063904310266176,
  'final_train_r2': 0.9828166824041141},
 'buddy_RNN': {'best_val_r2': 0.726462185382843,
  'final_train_r2': 0.8118318145473798},
 'cicero_RNN': {'best_val_r2': 0.6859094202518463,
  'final_train_r2': 0.8144127264618873},
 'gatsby_RNN': {'best_val_r2': 0.7927055309216181,
  'final_train_r2': 0.8883007996612124},
 'achilles_GRU': {'best_val_r2': 0.9257284142076969,
  'final_train_r2': 0.9883650825358927},
 'buddy_GRU': {'best_val_r2': 0.7638026848435402,
  'final_train_r2': 0.8716895145674547},
 'cicero_GRU': {'best_val_r2': 0.7520790249109268,
  'final_train_r2': 0.8665330655872822},
 'gatsby_GRU': {'best_val_r2': 0.8037704303860664,
  'final_train_r2': 0.9276171339054903}}

In [ ]:
"""
visualize_series1.py
════════════════════════════════════════════════════════
کد کامل visualization برای سری اول پروژه

پیش‌نیاز:
  - اجرای train_series1.py (یا run_all.py) قبل از این فایل
  - وجود فایل‌های results/series1/history_*.json
  - وجود فایل‌های results/series1/best_*.pt
  - وجود فایل‌های npz در همان پوشه

خروجی (در results/plots/):
  p1_gru_curves.png       ← منحنی‌های Loss و R² برای GRU
  p2_rnn_vs_gru.png       ← مقایسه RNN و GRU
  p3_comparison_table.png ← جدول + bar chart نهایی
  p4_scatter.png          ← Predicted vs True (scatter)
  p5_error_dist.png       ← توزیع خطای پیش‌بینی
  p6_timeseries.png       ← پیش‌بینی روی یک trial نمونه

اجرا:
  python visualize_series1.py
"""


# ════════════════════════════════════════════════════════
#  تنظیمات ثابت
# ════════════════════════════════════════════════════════
RATS = ['achilles', 'buddy', 'cicero', 'gatsby']

NEURON_COUNTS = {'achilles': 120, 'buddy': 48, 'cicero': 55, 'gatsby': 66}
TRIAL_COUNTS  = {'achilles': 85,  'buddy': 58, 'cicero': 93, 'gatsby': 85}

# رنگ اصلی هر موش
COLORS = {
    'achilles': '#2979FF',
    'buddy':    '#00BFA5',
    'cicero':   '#FF6D00',
    'gatsby':   '#AA00FF',
}

# پوشه‌های ورودی و خروجی
HIST_DIR  = 'results/series1'
MODEL_DIR = 'results/series1'
PLOT_DIR  = 'results/plots'
os.makedirs(PLOT_DIR, exist_ok=True)

# تم تیره برای همه نمودارها
BG_MAIN  = '#0F1117'
BG_PANEL = '#1A1E2E'
GRID_CLR = '#252535'
SPINE_CLR= '#333344'
TICK_CLR = '#888888'
LABEL_CLR= '#AAAAAA'
CLR_GRU  = '#FF6B6B'
CLR_RNN  = '#4ECDC4'


# ════════════════════════════════════════════════════════
#  مرحله ۰: بارگذاری history ها
# ════════════════════════════════════════════════════════
def load_histories():
    """بارگذاری همه فایل‌های history JSON"""
    h = {}
    for rat in RATS:
        for rnn in ['GRU', 'RNN']:
            path = os.path.join(HIST_DIR, f'history_{rat}_{rnn}.json')
            if not os.path.exists(path):
                raise FileNotFoundError(f'History not found: {path}\n'
                                        'ابتدا train_series1.py را اجرا کنید.')
            h[f'{rat}_{rnn}'] = json.load(open(path))
    print(f'✓ Histories loaded ({len(h)} models)')
    return h


# ════════════════════════════════════════════════════════
#  مرحله ۱: جمع‌آوری پیش‌بینی‌های validation
# ════════════════════════════════════════════════════════
def collect_predictions(force_recompute=False):
    """
    پیش‌بینی مدل‌های ذخیره‌شده روی validation set.
    اگر فایل pkl موجود باشد از آن استفاده می‌کند (سریع‌تر).
    force_recompute=True باعث محاسبه مجدد می‌شود.
    """
    pkl_path = os.path.join(HIST_DIR, 'val_predictions.pkl')

    if os.path.exists(pkl_path) and not force_recompute:
        preds = pickle.load(open(pkl_path, 'rb'))
        print(f'✓ Predictions loaded from cache ({pkl_path})')
        return preds

    print('Computing predictions...')
    preds = {}
    for rnn_type in ['GRU', 'RNN']:
        for rat in RATS:
            tag = f'{rat}_{rnn_type}'
            model_path = os.path.join(MODEL_DIR, f'best_{tag}.pt')
            if not os.path.exists(model_path):
                print(f'  ⚠ Model not found: {model_path}')
                continue

            torch.manual_seed(42)
            _, val_ds = get_rat_hippo(f'data\\hippo_nn_{rat}.npz', 500)
            val_loader = DataLoader(val_ds, batch_size=32,
                                    shuffle=False, collate_fn=collate_fn)

            M = NEURON_COUNTS[rat]
            model = RatModel(M, 64, 128, rnn_type)
            model.load_state_dict(torch.load(model_path, map_location='cpu'))
            model.eval()

            all_pred, all_true = [], []
            with torch.no_grad():
                for X_pad, Y_pad, mask, lengths in val_loader:
                    pred = model(X_pad, lengths)
                    all_pred.extend(pred[mask].numpy().tolist())
                    all_true.extend(Y_pad[mask].numpy().tolist())

            preds[tag] = {
                'pred': np.array(all_pred),
                'true': np.array(all_true),
            }
            print(f'  {tag}: {len(all_pred):,} timesteps')

    pickle.dump(preds, open(pkl_path, 'wb'))
    print(f'✓ Predictions saved to {pkl_path}')
    return preds


# ════════════════════════════════════════════════════════
#  ابزارهای کمکی برای استایل
# ════════════════════════════════════════════════════════
def style_ax(ax):
    """اعمال تم تیره به یک Axes"""
    ax.set_facecolor(BG_PANEL)
    ax.tick_params(colors=TICK_CLR, labelsize=8)
    ax.xaxis.label.set_color(LABEL_CLR)
    ax.yaxis.label.set_color(LABEL_CLR)
    ax.grid(color=GRID_CLR, lw=0.5)
    for sp in ax.spines.values():
        sp.set_color(SPINE_CLR)

def r2_rmse(pred, true):
    """محاسبه R² و RMSE"""
    pred = np.asarray(pred, dtype=np.float32)
    true = np.asarray(true, dtype=np.float32)
    ss_res = np.sum((true - pred) ** 2)
    ss_tot = np.sum((true - true.mean()) ** 2)
    r2   = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    rmse = np.sqrt(np.mean((true - pred) ** 2))
    return r2, rmse


# ════════════════════════════════════════════════════════
#  نمودار ۱: GRU Training Curves — Loss و R² (4 موش)
# ════════════════════════════════════════════════════════
def plot_gru_curves(histories):
    """
    ردیف بالا: MSE Loss (train/val) در طول epoch ها
    ردیف پایین: R² (train/val) در طول epoch ها
    خط عمودی نقطه‌چین = بهترین val R²
    """
    fig = plt.figure(figsize=(20, 9))
    fig.patch.set_facecolor(BG_MAIN)
    gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.30)
    fig.suptitle('Series 1 — GRU: Training Curves',
                 fontsize=16, fontweight='bold', color='white', y=0.98)

    for col, rat in enumerate(RATS):
        h   = histories[f'{rat}_GRU']
        eps = range(1, len(h['train_loss']) + 1)
        c   = COLORS[rat]

        # ── Loss ──────────────────────────────────────────
        ax = fig.add_subplot(gs[0, col])
        style_ax(ax)
        ax.plot(eps, h['train_loss'], color=CLR_GRU, lw=1.5, label='Train')
        ax.plot(eps, h['val_loss'],   color=CLR_RNN, lw=1.5, label='Val', ls='--')
        best_ep = int(np.argmin(h['val_loss'])) + 1
        ax.axvline(best_ep, color='white', lw=0.8, ls=':', alpha=0.45)
        ax.set_title(rat.capitalize(), color=c, fontsize=13,
                     fontweight='bold', pad=6)
        ax.set_xlabel('Epoch');  ax.set_ylabel('MSE Loss')
        if col == 0:
            ax.legend(fontsize=8, framealpha=0.2, labelcolor='white')

        # ── R² ────────────────────────────────────────────
        ax2 = fig.add_subplot(gs[1, col])
        style_ax(ax2)
        ax2.plot(eps, h['train_r2'], color=CLR_GRU, lw=1.5, label='Train')
        ax2.plot(eps, h['val_r2'],   color=CLR_RNN, lw=1.5, label='Val', ls='--')
        best_r2 = max(h['val_r2'])
        best_ep2 = int(np.argmax(h['val_r2'])) + 1
        ax2.axvline(best_ep2, color='white', lw=0.8, ls=':', alpha=0.45)
        ax2.axhline(best_r2,  color=c,       lw=0.8, ls=':', alpha=0.60)
        ax2.annotate(
            f'R²={best_r2:.3f}',
            xy=(best_ep2, best_r2),
            xytext=(best_ep2 + 4, best_r2 - 0.10),
            color=c, fontsize=8,
            arrowprops=dict(arrowstyle='->', color=c, lw=0.8),
        )
        ax2.set_xlabel('Epoch');  ax2.set_ylabel('R²')
        ax2.set_ylim([-0.3, 1.05])
        if col == 0:
            ax2.legend(fontsize=8, framealpha=0.2, labelcolor='white')

    out = os.path.join(PLOT_DIR, 'p1_gru_curves.png')
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')


# ════════════════════════════════════════════════════════
#  نمودار ۲: RNN vs GRU — Val R² و Train R²
# ════════════════════════════════════════════════════════
def plot_rnn_vs_gru(histories):
    """
    ردیف بالا: Val R² — GRU در مقابل RNN با ناحیه سایه
    ردیف پایین: Train R² مقایسه
    """
    fig = plt.figure(figsize=(20, 9))
    fig.patch.set_facecolor(BG_MAIN)
    gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.30)
    fig.suptitle('Series 1 — RNN vs GRU: Validation & Train R²',
                 fontsize=16, fontweight='bold', color='white', y=0.98)

    for col, rat in enumerate(RATS):
        h_g = histories[f'{rat}_GRU']
        h_r = histories[f'{rat}_RNN']
        eps = list(range(1, 61))
        c   = COLORS[rat]

        # ── Val R² ────────────────────────────────────────
        ax = fig.add_subplot(gs[0, col])
        style_ax(ax)
        ax.plot(eps, h_g['val_r2'], color=CLR_GRU, lw=2, label='GRU')
        ax.plot(eps, h_r['val_r2'], color=CLR_RNN, lw=2, label='RNN', ls='--')
        # ناحیه برتری GRU
        ax.fill_between(
            eps, h_r['val_r2'], h_g['val_r2'],
            where=[g > r for g, r in zip(h_g['val_r2'], h_r['val_r2'])],
            alpha=0.18, color=CLR_GRU, label='GRU advantage',
        )
        ax.set_title(rat.capitalize(), color=c, fontsize=13,
                     fontweight='bold', pad=6)
        ax.set_xlabel('Epoch');  ax.set_ylabel('Val R²')
        ax.set_ylim([-0.10, 1.05])
        ax.legend(fontsize=8, framealpha=0.2, labelcolor='white')

        # ── Train R² ──────────────────────────────────────
        ax2 = fig.add_subplot(gs[1, col])
        style_ax(ax2)
        ax2.plot(eps, h_g['train_r2'], color=CLR_GRU, lw=2, label='GRU')
        ax2.plot(eps, h_r['train_r2'], color=CLR_RNN, lw=2, label='RNN', ls='--')
        ax2.set_xlabel('Epoch');  ax2.set_ylabel('Train R²')
        ax2.set_ylim([-0.10, 1.05])
        ax2.legend(fontsize=8, framealpha=0.2, labelcolor='white')

    out = os.path.join(PLOT_DIR, 'p2_rnn_vs_gru.png')
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')


# ════════════════════════════════════════════════════════
#  نمودار ۳: Bar chart مقایسه نهایی + جدول آماری
# ════════════════════════════════════════════════════════
def plot_comparison_table(histories):
    """
    سمت چپ: grouped bar chart بهترین Val R²
    سمت راست: جدول کامل آمار (R², Train R², Δ)
    """
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.patch.set_facecolor(BG_MAIN)
    fig.suptitle('Series 1 — Final Performance Comparison',
                 fontsize=16, fontweight='bold', color='white')

    # ── Bar chart ─────────────────────────────────────────
    ax  = axes[0]
    ax.set_facecolor(BG_PANEL)
    x   = np.arange(len(RATS))
    w   = 0.35

    gru_val   = [max(histories[f'{r}_GRU']['val_r2'])   for r in RATS]
    rnn_val   = [max(histories[f'{r}_RNN']['val_r2'])   for r in RATS]

    bars1 = ax.bar(x - w/2, rnn_val, w, label='RNN  (Val R²)',
                   color=CLR_RNN, alpha=0.85, zorder=3)
    bars2 = ax.bar(x + w/2, gru_val, w, label='GRU  (Val R²)',
                   color=CLR_GRU, alpha=0.85, zorder=3)

    for bar, val in zip(bars1, rnn_val):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=10, color=CLR_RNN, fontweight='bold')
    for bar, val in zip(bars2, gru_val):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=10, color=CLR_GRU, fontweight='bold')

    for ref in [0.7, 0.8, 0.9]:
        ax.axhline(ref, color='white', lw=0.7, ls=':', alpha=0.25)

    ax.set_xticks(x)
    ax.set_xticklabels([r.capitalize() for r in RATS],
                       fontsize=12, color='white')
    ax.set_ylabel('Best Validation R²', color=LABEL_CLR, fontsize=11)
    ax.set_ylim([0.50, 1.05])
    ax.tick_params(colors=TICK_CLR)
    for sp in ax.spines.values(): sp.set_color(SPINE_CLR)
    ax.grid(axis='y', color=GRID_CLR, lw=0.5, zorder=0)
    ax.legend(fontsize=10, framealpha=0.2, labelcolor='white')
    ax.set_title('Best Validation R²: RNN vs GRU',
                 color='white', fontsize=12, pad=10)

    # ── جدول آماری ────────────────────────────────────────
    ax2 = axes[1]
    ax2.set_facecolor(BG_MAIN)
    ax2.axis('off')

    headers = ['Rat', 'Neurons', 'Trials',
               'RNN Val R²', 'GRU Val R²', 'Δ(GRU−RNN)',
               'RNN Train R²', 'GRU Train R²']
    rows = []
    for rat in RATS:
        g_v = max(histories[f'{rat}_GRU']['val_r2'])
        r_v = max(histories[f'{rat}_RNN']['val_r2'])
        g_t = max(histories[f'{rat}_GRU']['train_r2'])
        r_t = max(histories[f'{rat}_RNN']['train_r2'])
        delta = g_v - r_v
        rows.append([
            rat.capitalize(),
            str(NEURON_COUNTS[rat]),
            str(TRIAL_COUNTS[rat]),
            f'{r_v:.4f}',
            f'{g_v:.4f}',
            f'+{delta:.4f}' if delta >= 0 else f'{delta:.4f}',
            f'{r_t:.4f}',
            f'{g_t:.4f}',
        ])

    tbl = ax2.table(cellText=rows, colLabels=headers,
                    cellLoc='center', loc='center',
                    bbox=[0, 0.10, 1, 0.85])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)

    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor(SPINE_CLR)
        if r == 0:                          # header
            cell.set_facecolor('#2A2E45')
            cell.get_text().set_color('white')
            cell.get_text().set_fontweight('bold')
        elif r % 2 == 0:
            cell.set_facecolor('#1E2235')
            cell.get_text().set_color('#CCCCCC')
        else:
            cell.set_facecolor('#171B2C')
            cell.get_text().set_color('#CCCCCC')
        if r > 0 and c == 5:                # ستون Delta
            v = float(rows[r - 1][5])
            cell.get_text().set_color('#66FF99' if v >= 0 else '#FF6666')

    ax2.set_title('Detailed Statistics', color='white', fontsize=12, pad=10)

    out = os.path.join(PLOT_DIR, 'p3_comparison_table.png')
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')


# ════════════════════════════════════════════════════════
#  نمودار ۴: Scatter — Predicted vs True Position
# ════════════════════════════════════════════════════════
def plot_scatter(preds):
    """
    هر نقطه = یک timestep در validation.
    هر چه نقاط به خط قطری نزدیک‌تر باشند پیش‌بینی دقیق‌تر است.
    رنگ‌بندی با hexbin برای نمایش تراکم.
    """
    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    fig.patch.set_facecolor(BG_MAIN)
    fig.suptitle('Series 1 — Predicted vs True Position  (Validation Set)',
                 fontsize=16, fontweight='bold', color='white', y=0.99)

    for col, rat in enumerate(RATS):
        c = COLORS[rat]
        for row, rnn in enumerate(['GRU', 'RNN']):
            ax  = axes[row][col]
            ax.set_facecolor(BG_PANEL)
            tag = f'{rat}_{rnn}'
            p   = preds[tag]['pred']
            t   = preds[tag]['true']

            # نمایش تراکم با hexbin
            ax.hexbin(t, p, gridsize=40, cmap='plasma',
                      mincnt=1, extent=[0, 1.6, 0, 1.6])
            # خط perfect fit
            ax.plot([0, 1.6], [0, 1.6], 'w--', lw=1.2, alpha=0.7)

            r2, rmse = r2_rmse(p, t)
            ax.text(0.05, 1.48,
                    f'R²={r2:.4f}\nRMSE={rmse:.4f} m',
                    color='white', fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='#000000AA', edgecolor='none'))

            ax.set_title(f'{rat.capitalize()} — {rnn}',
                         color=c, fontsize=12, fontweight='bold')
            ax.set_xlabel('True position (m)', color=LABEL_CLR, fontsize=9)
            ax.set_ylabel('Predicted position (m)', color=LABEL_CLR, fontsize=9)
            ax.set_xlim([0, 1.6]);  ax.set_ylim([0, 1.6])
            ax.tick_params(colors=TICK_CLR, labelsize=8)
            for sp in ax.spines.values(): sp.set_color(SPINE_CLR)

    out = os.path.join(PLOT_DIR, 'p4_scatter.png')
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')


# ════════════════════════════════════════════════════════
#  نمودار ۵: Error Distribution — هیستوگرام خطا
# ════════════════════════════════════════════════════════
def plot_error_dist(preds):
    """
    توزیع خطای پیش‌بینی (pred − true) برای GRU و RNN.
    توزیع باریک و متقارن حول صفر = پیش‌بینی خوب.
    σ = انحراف معیار خطا.
    """
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.patch.set_facecolor(BG_MAIN)
    fig.suptitle('Series 1 — Prediction Error Distribution  (GRU vs RNN)',
                 fontsize=15, fontweight='bold', color='white', y=1.02)

    for col, rat in enumerate(RATS):
        ax = axes[col]
        style_ax(ax)
        c  = COLORS[rat]

        for rnn, clr in [('GRU', CLR_GRU), ('RNN', CLR_RNN)]:
            p   = np.asarray(preds[f'{rat}_{rnn}']['pred'], dtype=np.float32)
            t   = np.asarray(preds[f'{rat}_{rnn}']['true'], dtype=np.float32)
            err = p - t
            ax.hist(err, bins=60, alpha=0.55, color=clr, density=True,
                    label=f'{rnn}  σ={err.std():.3f}m')
            ax.axvline(err.mean(), color=clr, lw=1.5, ls=':')

        ax.axvline(0, color='white', lw=1.0, ls='--', alpha=0.45)
        ax.set_title(rat.capitalize(), color=c, fontsize=13, fontweight='bold')
        ax.set_xlabel('Error  (pred − true)  [m]', color=LABEL_CLR, fontsize=10)
        ax.set_ylabel('Density', color=LABEL_CLR, fontsize=10)
        ax.legend(fontsize=9, framealpha=0.2, labelcolor='white')

    out = os.path.join(PLOT_DIR, 'p5_error_dist.png')
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')


# ════════════════════════════════════════════════════════
#  نمودار ۶: Time-series — پیش‌بینی در طول زمان
# ════════════════════════════════════════════════════════
def plot_timeseries(preds, n_timesteps=250):
    """
    نمایش n_timesteps اول از validation set هر موش.
    خط سفید = موقعیت واقعی موش.
    خط رنگی = پیش‌بینی مدل.
    ناحیه رنگی = خطای لحظه‌ای.
    """
    fig, axes = plt.subplots(4, 2, figsize=(18, 14))
    fig.patch.set_facecolor(BG_MAIN)
    fig.suptitle(f'Series 1 — Sample Trajectory: Predicted vs True  '
                 f'(first {n_timesteps} timesteps of validation)',
                 fontsize=15, fontweight='bold', color='white', y=1.0)

    for row, rat in enumerate(RATS):
        c = COLORS[rat]
        for col, rnn in enumerate(['GRU', 'RNN']):
            ax  = axes[row][col]
            style_ax(ax)
            tag = f'{rat}_{rnn}'
            p   = preds[tag]['pred']
            t   = preds[tag]['true']
            N   = min(n_timesteps, len(p))
            t_ax = np.arange(N)

            ax.plot(t_ax, t[:N], color='white', lw=1.8,
                    alpha=0.9, label='True position')
            ax.plot(t_ax, p[:N], color=c, lw=1.5,
                    alpha=0.85, label=f'{rnn} predicted')
            ax.fill_between(t_ax, t[:N], p[:N], alpha=0.18, color=c)

            r2, rmse = r2_rmse(p, t)
            ax.text(0.02, 0.06, f'R²={r2:.4f}',
                    transform=ax.transAxes, color=c,
                    fontsize=10, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.25',
                              facecolor='#000000AA', edgecolor='none'))

            ax.set_ylim([0, 1.6])
            ax.set_xlabel('Timestep', color=LABEL_CLR, fontsize=9)
            ax.set_ylabel('Position (m)', color=LABEL_CLR, fontsize=9)
            ax.set_title(f'{rat.capitalize()} — {rnn}',
                         color=c, fontsize=11, fontweight='bold')
            ax.legend(fontsize=8, framealpha=0.2, labelcolor='white',
                      loc='upper right')

    out = os.path.join(PLOT_DIR, 'p6_timeseries.png')
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')


# ════════════════════════════════════════════════════════
#  اجرای اصلی
# ════════════════════════════════════════════════════════
if __name__ == '__main__':
    print('\n' + '═' * 52)
    print('  Series 1 — Visualization')
    print('═' * 52 + '\n')

    # بارگذاری داده‌ها
    histories = load_histories()
    preds     = collect_predictions(force_recompute=False)

    # رسم نمودارها
    print('\nGenerating plots...')
    plot_gru_curves(histories)        # نمودار ۱
    plot_rnn_vs_gru(histories)        # نمودار ۲
    plot_comparison_table(histories)  # نمودار ۳
    plot_scatter(preds)               # نمودار ۴
    plot_error_dist(preds)            # نمودار ۵
    plot_timeseries(preds)            # نمودار ۶

    print(f'\n✓ All 6 plots saved to: {PLOT_DIR}/')
    print('\nفایل‌های خروجی:')
    for i, name in enumerate([
        'p1_gru_curves.png       ← Loss و R² در طول آموزش (GRU)',
        'p2_rnn_vs_gru.png       ← مقایسه RNN و GRU',
        'p3_comparison_table.png ← جدول + bar chart نهایی',
        'p4_scatter.png          ← Predicted vs True (scatter)',
        'p5_error_dist.png       ← توزیع خطا',
        'p6_timeseries.png       ← پیش‌بینی در طول زمان',
    ], 1):
        print(f'  {i}. {name}')


════════════════════════════════════════════════════
  Series 1 — Visualization
════════════════════════════════════════════════════

✓ Histories loaded (8 models)
Computing predictions...
  [data\hippo_nn_achilles.npz] trials=85 neurons=120 train=68 val=17
  achilles_GRU: 2,134 timesteps
  [data\hippo_nn_buddy.npz] trials=58 neurons=48 train=46 val=12
  buddy_GRU: 1,457 timesteps
  [data\hippo_nn_cicero.npz] trials=93 neurons=55 train=74 val=19
  cicero_GRU: 5,847 timesteps
  [data\hippo_nn_gatsby.npz] trials=85 neurons=66 train=68 val=17
  gatsby_GRU: 3,521 timesteps
  [data\hippo_nn_achilles.npz] trials=85 neurons=120 train=68 val=17
  achilles_RNN: 2,134 timesteps
  [data\hippo_nn_buddy.npz] trials=58 neurons=48 train=46 val=12
  buddy_RNN: 1,457 timesteps
  [data\hippo_nn_cicero.npz] trials=93 neurons=55 train=74 val=19
  cicero_RNN: 5,847 timesteps
  [data\hippo_nn_gatsby.npz] trials=85 neurons=66 train=68 val=17
  gatsby_RNN: 3,521 timesteps
✓ Predictions saved to results/serie

In [ ]:
def run_series2():
    print('\n' + '█'*55)
    print('  SERIES 2 — Shared RNN (all 4 rats)')
    print('█'*55)

    torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])
    t0 = time.time()

    print('\nLoading datasets...')
    tr_l, vl_l = make_loaders(list(RATS.keys()), CFG['batch_size'])

    rat_dims = {r: RATS[r]['M'] for r in RATS}
    model = SharedRNNModel(
        rat_dims=rat_dims,
        latent_dim=CFG['latent_dim'],
        hidden_dim=CFG['hidden_dim'],
        rnn_type='GRU',
        num_layers=CFG['num_layers'],
    ).to(DEVICE)
    print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

    opt  = torch.optim.Adam(model.parameters(),
                            lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sch  = make_scheduler(opt)
    hist = {r: {'train_loss':[], 'train_r2':[], 'val_loss':[], 'val_r2':[]}
            for r in RATS}
    best_avg = -np.inf
    best_path = 'results/series2/best_shared.pt'

    for ep in range(1, CFG['epochs_s2'] + 1):
        tr_stats = train_one_epoch_shared(model, tr_l, opt, DEVICE)
        vl_stats = eval_one_epoch_shared(model, vl_l, DEVICE)

        avg_vl = 0.0
        for rat in RATS:
            hist[rat]['train_loss'].append(tr_stats[rat][0])
            hist[rat]['train_r2'].append(tr_stats[rat][1])
            hist[rat]['val_loss'].append(vl_stats[rat][0])
            hist[rat]['val_r2'].append(vl_stats[rat][1])
            avg_vl += vl_stats[rat][1]
        avg_vl /= len(RATS)
        sch.step(avg_vl)

        if avg_vl > best_avg:
            best_avg = avg_vl
            torch.save(model.state_dict(), best_path)

        if ep % 10 == 0 or ep == 1:
            print(f'\n  Epoch {ep:3d}/{CFG["epochs_s2"]}  (t={time.time()-t0:.0f}s)')
            for rat in RATS:
                print(f'    {rat:8s}: Tr R²={tr_stats[rat][1]:.4f}  Vl R²={vl_stats[rat][1]:.4f}')
            print(f'    AvgVl R²={avg_vl:.4f}  Best={best_avg:.4f}')

    save_json(hist, 'results/series2/history_shared.json')
    summary = {r: {'best_val_r2': max(hist[r]['val_r2']),
                   'final_train_r2': hist[r]['train_r2'][-1]} for r in RATS}
    save_json(summary, 'results/series2/summary.json')

    print('\n  ── Series 2 Summary ──')
    for r, s in summary.items():
        print(f'  {r:8s}: Val R²={s["best_val_r2"]:.4f}')
    return model, hist, summary

In [ ]:
def run_series3():
    print('\n' + '█'*55)
    print('  SERIES 3 — Transfer Learning (gatsby)')
    print('█'*55)

    torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])

    # ── فاز اول: pretrain روی ۳ موش ─────────────────────
    print('\n  Phase 1: Pre-training on achilles, buddy, cicero')
    t0 = time.time()

    src_info = {r: RATS[r] for r in SOURCE_RATS}
    tr_l, vl_l = make_loaders(SOURCE_RATS, CFG['batch_size'])

    rat_dims = {r: RATS[r]['M'] for r in SOURCE_RATS}
    model = SharedRNNModel(
        rat_dims=rat_dims,
        latent_dim=CFG['latent_dim'],
        hidden_dim=CFG['hidden_dim'],
        rnn_type='GRU',
        num_layers=CFG['num_layers'],
    ).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(),
                           lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sch = make_scheduler(opt)
    pre_hist = {r: {'train_loss':[], 'train_r2':[], 'val_loss':[], 'val_r2':[]}
                for r in SOURCE_RATS}
    best_pre = -np.inf
    pre_path = 'results/series3/pretrained.pt'

    for ep in range(1, CFG['epochs_pre'] + 1):
        tr_s = train_one_epoch_shared(model, tr_l, opt, DEVICE)
        vl_s = eval_one_epoch_shared(model, vl_l, DEVICE)
        avg  = 0.0
        for r in SOURCE_RATS:
            pre_hist[r]['train_loss'].append(tr_s[r][0])
            pre_hist[r]['train_r2'].append(tr_s[r][1])
            pre_hist[r]['val_loss'].append(vl_s[r][0])
            pre_hist[r]['val_r2'].append(vl_s[r][1])
            avg += vl_s[r][1]
        avg /= len(SOURCE_RATS)
        sch.step(avg)
        if avg > best_pre:
            best_pre = avg
            torch.save(model.state_dict(), pre_path)
        if ep % 10 == 0 or ep == 1:
            print(f'  Ep {ep:3d}/{CFG["epochs_pre"]}  AvgVl R²={avg:.4f}  Best={best_pre:.4f}  t={time.time()-t0:.0f}s')

    save_json(pre_hist, 'results/series3/history_pretrain.json')
    print(f'  ✓ Pre-trained. Best Avg Val R²={best_pre:.4f}')

    # ── فاز دوم: fine-tune Encoder جدید برای gatsby ──────
    print(f'\n  Phase 2: Fine-tuning new Encoder for "{TARGET_RAT}"')
    t1 = time.time()

    # بارگذاری بهترین وزن‌ها
    model.load_state_dict(torch.load(pre_path, map_location=DEVICE))

    # فریز RNN + Regressor
    model.freeze_shared()

    # اضافه کردن Encoder جدید برای gatsby
    model.add_rat_encoder(TARGET_RAT, RATS[TARGET_RAT]['M'])
    model = model.to(DEVICE)

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total     = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {n_trainable:,} / Total: {n_total:,}')

    tgt_tr, tgt_vl = make_loaders([TARGET_RAT], CFG['batch_size'])
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    opt2 = torch.optim.Adam(trainable_params,
                            lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sch2 = make_scheduler(opt2)

    ft_hist = {'train_loss':[], 'train_r2':[], 'val_loss':[], 'val_r2':[]}
    best_ft = -np.inf
    ft_path = 'results/series3/finetuned.pt'

    for ep in range(1, CFG['epochs_ft'] + 1):
        tr_s = train_one_epoch_shared(model, tgt_tr, opt2, DEVICE)
        vl_s = eval_one_epoch_shared(model, tgt_vl, DEVICE)
        tr_loss, tr_r2 = tr_s[TARGET_RAT]
        vl_loss, vl_r2 = vl_s[TARGET_RAT]
        sch2.step(vl_r2)

        ft_hist['train_loss'].append(tr_loss)
        ft_hist['train_r2'].append(tr_r2)
        ft_hist['val_loss'].append(vl_loss)
        ft_hist['val_r2'].append(vl_r2)

        if vl_r2 > best_ft:
            best_ft = vl_r2
            torch.save(model.state_dict(), ft_path)

        if ep % 10 == 0 or ep == 1:
            log_row(ep, CFG['epochs_ft'], tr_loss, tr_r2, vl_loss, vl_r2, best_ft, t1)

    save_json(ft_hist, 'results/series3/history_finetune.json')
    summary = {
        'pretrain': {r: {'best_val_r2': max(pre_hist[r]['val_r2'])} for r in SOURCE_RATS},
        'finetune_gatsby': {'best_val_r2': best_ft,
                            'final_train_r2': ft_hist['train_r2'][-1]}
    }
    save_json(summary, 'results/series3/summary.json')

    print(f'  ✓ Fine-tuned gatsby. Best Val R²={best_ft:.4f}')
    return ft_hist, summary
